# Image Preprocessing Pipeline - Reusable Templates

Pipeline preprocessing yang komprehensif dan mudah digunakan untuk berbagai kasus:
- Binary Classification
- Multi-class Classification
- Multi-label Classification
- Object Detection
- Segmentation

Semua fungsi dirancang untuk reusable dan mudah disesuaikan.

In [ ]:
import cv2
import numpy as np
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
import torchvision.transforms as transforms
from pathlib import Path
import matplotlib.pyplot as plt
from typing import Tuple, List, Optional
import warnings
warnings.filterwarnings('ignore')

## 1. Basic Preprocessing Functions

Fungsi dasar untuk preprocessing gambar

In [ ]:
def load_image(path: str, mode: str = 'RGB') -> np.ndarray:
    """Load image from path"""
    if mode == 'RGB':
        img = cv2.imread(str(path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    elif mode == 'GRAY':
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    else:
        img = cv2.imread(str(path))
    return img

def resize_image(img: np.ndarray, size: Tuple[int, int]) -> np.ndarray:
    """Resize image to target size"""
    return cv2.resize(img, size, interpolation=cv2.INTER_AREA)

def normalize_image(img: np.ndarray, method: str = 'standard') -> np.ndarray:
    """Normalize image pixels"""
    if method == 'standard':
        return img / 255.0
    elif method == 'mean_std':
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        return (img / 255.0 - mean) / std
    elif method == 'minmax':
        return (img - img.min()) / (img.max() - img.min())
    return img

def denormalize_image(img: np.ndarray, method: str = 'standard') -> np.ndarray:
    """Denormalize image for visualization"""
    if method == 'mean_std':
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = img * std + mean
    return (img * 255).clip(0, 255).astype(np.uint8)

print("Basic preprocessing functions loaded")

## 2. Advanced Preprocessing Functions

Fungsi preprocessing lanjutan untuk enhancement

In [ ]:
def apply_clahe(img: np.ndarray, clip_limit: float = 2.0, tile_size: Tuple[int, int] = (8, 8)) -> np.ndarray:
    """Apply CLAHE for contrast enhancement"""
    if len(img.shape) == 2:
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
        return clahe.apply(img)
    else:
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def histogram_equalization(img: np.ndarray) -> np.ndarray:
    """Apply histogram equalization"""
    if len(img.shape) == 2:
        return cv2.equalizeHist(img)
    else:
        ycrcb = cv2.cvtColor(img, cv2.COLOR_RGB2YCrCb)
        ycrcb[:, :, 0] = cv2.equalizeHist(ycrcb[:, :, 0])
        return cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2RGB)

def adjust_brightness_contrast(img: np.ndarray, brightness: int = 0, contrast: int = 0) -> np.ndarray:
    """Adjust brightness and contrast"""
    img = img.astype(np.int16)
    img = img * (contrast / 127 + 1) - contrast + brightness
    img = np.clip(img, 0, 255)
    return img.astype(np.uint8)

def denoise_image(img: np.ndarray, method: str = 'gaussian') -> np.ndarray:
    """Apply denoising"""
    if method == 'gaussian':
        return cv2.GaussianBlur(img, (5, 5), 0)
    elif method == 'median':
        return cv2.medianBlur(img, 5)
    elif method == 'bilateral':
        return cv2.bilateralFilter(img, 9, 75, 75)
    elif method == 'nlm':
        return cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)
    return img

def sharpen_image(img: np.ndarray) -> np.ndarray:
    """Sharpen image"""
    kernel = np.array([[-1, -1, -1], [-1, 9, -1], [-1, -1, -1]])
    return cv2.filter2D(img, -1, kernel)

print("Advanced preprocessing functions loaded")

## 3. Augmentation Pipelines

Pipeline augmentasi untuk training

In [ ]:
def get_augmentation_pipeline(mode: str = 'light', img_size: Tuple[int, int] = (224, 224)):
    """
    Get augmentation pipeline for different scenarios
    
    Args:
        mode: 'light', 'medium', 'heavy', 'custom'
        img_size: target image size
    """
    
    if mode == 'light':
        return A.Compose([
            A.Resize(img_size[0], img_size[1]),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=10, p=0.5),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])
    
    elif mode == 'medium':
        return A.Compose([
            A.Resize(img_size[0], img_size[1]),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.Rotate(limit=20, p=0.5),
            A.RandomBrightnessContrast(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])
    
    elif mode == 'heavy':
        return A.Compose([
            A.Resize(img_size[0], img_size[1]),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Rotate(limit=30, p=0.7),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.7),
            A.ShiftScaleRotate(shift_limit=0.15, scale_limit=0.15, rotate_limit=20, p=0.7),
            A.OneOf([
                A.GaussNoise(p=1.0),
                A.GaussianBlur(p=1.0),
                A.MotionBlur(p=1.0)
            ], p=0.5),
            A.OneOf([
                A.OpticalDistortion(p=1.0),
                A.GridDistortion(p=1.0),
                A.ElasticTransform(p=1.0)
            ], p=0.3),
            A.HueSaturationValue(p=0.5),
            A.RGBShift(p=0.5),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])
    
    else:
        return A.Compose([
            A.Resize(img_size[0], img_size[1]),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

def get_validation_pipeline(img_size: Tuple[int, int] = (224, 224)):
    """Validation/test pipeline without augmentation"""
    return A.Compose([
        A.Resize(img_size[0], img_size[1]),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])

print("Augmentation pipelines loaded")

## 4. PyTorch Transform Pipelines

Pipeline menggunakan torchvision transforms

In [ ]:
def get_torch_train_transforms(img_size: Tuple[int, int] = (224, 224)):
    """PyTorch training transforms"""
    return transforms.Compose([
        transforms.Resize((img_size[0], img_size[1])),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def get_torch_val_transforms(img_size: Tuple[int, int] = (224, 224)):
    """PyTorch validation transforms"""
    return transforms.Compose([
        transforms.Resize((img_size[0], img_size[1])),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def get_torch_test_transforms(img_size: Tuple[int, int] = (224, 224)):
    """PyTorch test transforms with TTA"""
    return transforms.Compose([
        transforms.Resize((img_size[0], img_size[1])),
        transforms.FiveCrop(img_size[0]),
        transforms.Lambda(lambda crops: torch.stack([
            transforms.ToTensor()(crop) for crop in crops
        ])),
        transforms.Lambda(lambda tensors: torch.stack([
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(t) 
            for t in tensors
        ]))
    ])

print("PyTorch transform pipelines loaded")

## 5. Preprocessing Presets

Preset konfigurasi untuk berbagai use case

In [ ]:
class PreprocessingPreset:
    """Preset configurations for different use cases"""
    
    @staticmethod
    def binary_classification(img_size=(224, 224)):
        """Binary classification preset"""
        return {
            'train': get_augmentation_pipeline('medium', img_size),
            'val': get_validation_pipeline(img_size),
            'description': 'Balanced augmentation for binary tasks'
        }
    
    @staticmethod
    def multiclass_classification(img_size=(224, 224), n_classes=10):
        """Multi-class classification preset"""
        if n_classes < 10:
            mode = 'heavy'
        elif n_classes < 100:
            mode = 'medium'
        else:
            mode = 'light'
        return {
            'train': get_augmentation_pipeline(mode, img_size),
            'val': get_validation_pipeline(img_size),
            'description': f'Augmentation adapted for {n_classes} classes'
        }
    
    @staticmethod
    def multilabel_classification(img_size=(224, 224)):
        """Multi-label classification preset"""
        return {
            'train': get_augmentation_pipeline('light', img_size),
            'val': get_validation_pipeline(img_size),
            'description': 'Conservative augmentation for multi-label'
        }
    
    @staticmethod
    def small_dataset(img_size=(224, 224)):
        """Heavy augmentation for small datasets"""
        return {
            'train': get_augmentation_pipeline('heavy', img_size),
            'val': get_validation_pipeline(img_size),
            'description': 'Heavy augmentation for data scarcity'
        }
    
    @staticmethod
    def medical_imaging(img_size=(224, 224)):
        """Preset for medical images"""
        return {
            'train': A.Compose([
                A.Resize(img_size[0], img_size[1]),
                A.HorizontalFlip(p=0.5),
                A.Rotate(limit=10, p=0.5),
                A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ]),
            'val': get_validation_pipeline(img_size),
            'description': 'Conservative for medical accuracy'
        }
    
    @staticmethod
    def document_ocr(img_size=(224, 224)):
        """Preset for document/OCR tasks"""
        return {
            'train': A.Compose([
                A.Resize(img_size[0], img_size[1]),
                A.Rotate(limit=5, p=0.5),
                A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.7),
                A.GaussNoise(var_limit=(5.0, 20.0), p=0.3),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ]),
            'val': get_validation_pipeline(img_size),
            'description': 'Document-specific augmentation'
        }

print("Preprocessing presets loaded")

## 6. Custom Preprocessing Pipeline Class

Class untuk membuat pipeline custom yang reusable

In [ ]:
class ImagePreprocessor:
    """Flexible image preprocessor class"""
    
    def __init__(self, img_size=(224, 224), mode='RGB'):
        self.img_size = img_size
        self.mode = mode
        self.pipeline = []
    
    def add_step(self, func, **kwargs):
        """Add preprocessing step"""
        self.pipeline.append((func, kwargs))
        return self
    
    def process(self, img_path):
        """Process image through pipeline"""
        img = load_image(img_path, self.mode)
        
        for func, kwargs in self.pipeline:
            img = func(img, **kwargs)
        
        return img
    
    def reset(self):
        """Reset pipeline"""
        self.pipeline = []
        return self
    
    def get_pipeline_info(self):
        """Get information about current pipeline"""
        info = []
        for func, kwargs in self.pipeline:
            info.append(f"{func.__name__}: {kwargs}")
        return info

def create_custom_pipeline():
    """Example: Create custom pipeline"""
    processor = ImagePreprocessor(img_size=(224, 224))
    processor.add_step(resize_image, size=(224, 224))
    processor.add_step(apply_clahe, clip_limit=2.0)
    processor.add_step(denoise_image, method='gaussian')
    processor.add_step(normalize_image, method='standard')
    return processor

print("Custom preprocessor class loaded")

## 7. Batch Processing

Fungsi untuk memproses batch images

In [ ]:
def batch_process_images(image_paths: List[str], pipeline, output_dir: Optional[str] = None):
    """Process multiple images"""
    processed = []
    
    for img_path in image_paths:
        img = load_image(img_path)
        
        if hasattr(pipeline, 'process'):
            result = pipeline.process(img_path)
        else:
            result = pipeline(image=img)['image']
        
        processed.append(result)
        
        if output_dir:
            output_path = Path(output_dir) / Path(img_path).name
            if isinstance(result, torch.Tensor):
                result = result.permute(1, 2, 0).numpy()
                result = denormalize_image(result)
            cv2.imwrite(str(output_path), cv2.cvtColor(result, cv2.COLOR_RGB2BGR))
    
    return processed

def process_directory(input_dir: str, output_dir: str, pipeline, recursive: bool = False):
    """Process all images in directory"""
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True, parents=True)
    
    if recursive:
        image_files = list(input_path.rglob('*.[jp][pn]g')) + list(input_path.rglob('*.jpeg'))
    else:
        image_files = list(input_path.glob('*.[jp][pn]g')) + list(input_path.glob('*.jpeg'))
    
    return batch_process_images([str(f) for f in image_files], pipeline, str(output_path))

print("Batch processing functions loaded")

## 8. Visualization Utilities

Fungsi untuk visualisasi hasil preprocessing

In [ ]:
def show_preprocessing_comparison(img_path: str, pipelines: dict, figsize=(15, 5)):
    """Compare different preprocessing pipelines"""
    original = load_image(img_path)
    
    n_pipelines = len(pipelines) + 1
    fig, axes = plt.subplots(1, n_pipelines, figsize=figsize)
    
    axes[0].imshow(original)
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    for idx, (name, pipeline) in enumerate(pipelines.items(), 1):
        if hasattr(pipeline, 'process'):
            processed = pipeline.process(img_path)
        else:
            processed = pipeline(image=original)['image']
        
        if isinstance(processed, torch.Tensor):
            processed = processed.permute(1, 2, 0).numpy()
            processed = denormalize_image(processed)
        
        axes[idx].imshow(processed)
        axes[idx].set_title(name)
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

def show_augmentation_samples(img_path: str, pipeline, n_samples=6, figsize=(15, 10)):
    """Show multiple augmentation samples"""
    img = load_image(img_path)
    
    rows = (n_samples + 2) // 3
    fig, axes = plt.subplots(rows, 3, figsize=figsize)
    axes = axes.ravel()
    
    for idx in range(n_samples):
        augmented = pipeline(image=img)['image']
        
        if isinstance(augmented, torch.Tensor):
            augmented = augmented.permute(1, 2, 0).numpy()
            augmented = denormalize_image(augmented)
        
        axes[idx].imshow(augmented)
        axes[idx].set_title(f'Sample {idx+1}')
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

def plot_image_statistics(img: np.ndarray):
    """Plot image statistics"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    axes[0, 0].imshow(img)
    axes[0, 0].set_title('Image')
    axes[0, 0].axis('off')
    
    if len(img.shape) == 3:
        for i, color in enumerate(['Red', 'Green', 'Blue']):
            axes[0, 1].hist(img[:, :, i].ravel(), bins=256, alpha=0.5, label=color)
        axes[0, 1].legend()
    else:
        axes[0, 1].hist(img.ravel(), bins=256, color='gray')
    axes[0, 1].set_title('Histogram')
    axes[0, 1].set_xlabel('Pixel Value')
    axes[0, 1].set_ylabel('Frequency')
    
    axes[1, 0].text(0.1, 0.9, f'Shape: {img.shape}', transform=axes[1, 0].transAxes)
    axes[1, 0].text(0.1, 0.8, f'Dtype: {img.dtype}', transform=axes[1, 0].transAxes)
    axes[1, 0].text(0.1, 0.7, f'Min: {img.min():.2f}', transform=axes[1, 0].transAxes)
    axes[1, 0].text(0.1, 0.6, f'Max: {img.max():.2f}', transform=axes[1, 0].transAxes)
    axes[1, 0].text(0.1, 0.5, f'Mean: {img.mean():.2f}', transform=axes[1, 0].transAxes)
    axes[1, 0].text(0.1, 0.4, f'Std: {img.std():.2f}', transform=axes[1, 0].transAxes)
    axes[1, 0].set_title('Statistics')
    axes[1, 0].axis('off')
    
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()

print("Visualization utilities loaded")

## 9. Usage Examples

Contoh penggunaan untuk berbagai skenario

In [ ]:
usage_guide = """
USAGE GUIDE
===========

1. BINARY CLASSIFICATION
-------------------------
preset = PreprocessingPreset.binary_classification(img_size=(224, 224))
train_transform = preset['train']
val_transform = preset['val']

# Apply to image
img = load_image('image.jpg')
augmented = train_transform(image=img)['image']


2. MULTI-CLASS CLASSIFICATION
------------------------------
preset = PreprocessingPreset.multiclass_classification(img_size=(224, 224), n_classes=20)
train_transform = preset['train']
val_transform = preset['val']


3. MULTI-LABEL CLASSIFICATION
------------------------------
preset = PreprocessingPreset.multilabel_classification(img_size=(224, 224))
train_transform = preset['train']
val_transform = preset['val']


4. SMALL DATASET (Heavy Augmentation)
--------------------------------------
preset = PreprocessingPreset.small_dataset(img_size=(224, 224))
train_transform = preset['train']


5. CUSTOM PIPELINE
------------------
processor = ImagePreprocessor(img_size=(224, 224))
processor.add_step(resize_image, size=(224, 224))
processor.add_step(apply_clahe, clip_limit=2.0)
processor.add_step(denoise_image, method='gaussian')
processed = processor.process('image.jpg')


6. PYTORCH TRANSFORMS
---------------------
train_transforms = get_torch_train_transforms(img_size=(224, 224))
val_transforms = get_torch_val_transforms(img_size=(224, 224))

# With PIL Image
from PIL import Image
pil_img = Image.open('image.jpg')
tensor = train_transforms(pil_img)


7. ALBUMENTATIONS
-----------------
pipeline = get_augmentation_pipeline(mode='medium', img_size=(224, 224))
img = load_image('image.jpg')
augmented = pipeline(image=img)['image']


8. BATCH PROCESSING
-------------------
image_paths = ['img1.jpg', 'img2.jpg', 'img3.jpg']
pipeline = get_validation_pipeline(img_size=(224, 224))
results = batch_process_images(image_paths, pipeline, output_dir='processed/')


9. MEDICAL IMAGING
------------------
preset = PreprocessingPreset.medical_imaging(img_size=(224, 224))
train_transform = preset['train']


10. DOCUMENT/OCR
----------------
preset = PreprocessingPreset.document_ocr(img_size=(224, 224))
train_transform = preset['train']


VISUALIZATION
-------------
# Compare preprocessing methods
pipelines = {
    'CLAHE': lambda img: apply_clahe(img),
    'Histogram EQ': lambda img: histogram_equalization(img),
    'Denoise': lambda img: denoise_image(img, 'gaussian')
}
show_preprocessing_comparison('image.jpg', pipelines)

# Show augmentation samples
pipeline = get_augmentation_pipeline('heavy')
show_augmentation_samples('image.jpg', pipeline, n_samples=6)
"""

print(usage_guide)

### Example 1: Binary Classification (Cat vs Dog)

In [ ]:
# Binary Classification Example
from torch.utils.data import Dataset, DataLoader
import torch

class BinaryDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img = load_image(self.image_paths[idx])
        
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented['image']
        
        # Convert to tensor and normalize
        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        label = self.labels[idx]
        
        return img, label

# Setup preprocessing
preset = PreprocessingPreset.binary_classification(img_size=(224, 224))
train_transform = preset['train']
val_transform = preset['val']

# Example usage
# train_dataset = BinaryDataset(train_paths, train_labels, transform=train_transform)
# val_dataset = BinaryDataset(val_paths, val_labels, transform=val_transform)
# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Binary Classification Preset Created!")
print(f"Train Transform: {train_transform}")
print(f"Val Transform: {val_transform}")

### Example 2: Multi-Class Classification (Aksara Jawa)

In [ ]:
# Multi-Class Classification for Aksara Jawa (20 classes)
from torchvision import datasets

# Setup preprocessing for 20 classes (Aksara Jawa characters)
preset = PreprocessingPreset.multiclass_classification(img_size=(224, 224), n_classes=20)
train_transform = preset['train']
val_transform = preset['val']

# Create custom dataset with Albumentations
class MultiClassDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        
        # Get all image paths and labels
        self.image_paths = []
        self.labels = []
        self.classes = sorted(os.listdir(root_dir))
        
        for idx, class_name in enumerate(self.classes):
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                        self.image_paths.append(os.path.join(class_dir, img_name))
                        self.labels.append(idx)
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img = load_image(self.image_paths[idx])
        
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented['image']
        
        # Convert to tensor
        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        label = self.labels[idx]
        
        return img, label

# Example usage for Aksara Jawa
# train_dataset = MultiClassDataset('dataset-aksara-jawa/train', transform=train_transform)
# val_dataset = MultiClassDataset('dataset-aksara-jawa/val', transform=val_transform)

print("Multi-Class Classification Preset Created!")
print(f"Number of classes: 20 (Aksara Jawa characters)")
print(f"Train augmentations: {len(train_transform.transforms)} transforms")
print(f"Val augmentations: {len(val_transform.transforms)} transforms")

### Example 3: Custom Pipeline for Specific Needs

In [ ]:
# Custom Pipeline Example: Historical Document Processing
processor = ImagePreprocessor(img_size=(224, 224))

# Add preprocessing steps
processor.add_step(resize_image, size=(224, 224))
processor.add_step(apply_clahe, clip_limit=3.0, tile_grid_size=(8, 8))
processor.add_step(denoise_image, method='bilateral', d=9)
processor.add_step(sharpen_image)

# Process single image
# processed_img = processor.process('historical_document.jpg')

# Process batch
# image_paths = ['doc1.jpg', 'doc2.jpg', 'doc3.jpg']
# results = processor.process_batch(image_paths, save_dir='processed_docs/')

print("Custom Pipeline Created!")
print(f"Pipeline steps: {processor.steps}")
print("\nCustom pipeline is perfect for:")
print("- Historical document restoration")
print("- Low quality image enhancement")
print("- Specialized preprocessing needs")


# Another Example: Medical X-Ray Enhancement
medical_processor = ImagePreprocessor(img_size=(512, 512))
medical_processor.add_step(resize_image, size=(512, 512))
medical_processor.add_step(apply_clahe, clip_limit=2.5)
medical_processor.add_step(histogram_equalization)
medical_processor.add_step(denoise_image, method='non_local_means')

print("\n\nMedical Image Pipeline Created!")
print(f"Image size: 512x512 (higher resolution for medical imaging)")
print(f"Number of preprocessing steps: {len(medical_processor.steps)}")

### Example 4: Small Dataset with Heavy Augmentation

In [ ]:
# Small Dataset Example (< 1000 images per class)
# Use heavy augmentation to create more training samples

preset = PreprocessingPreset.small_dataset(img_size=(224, 224))
heavy_train_transform = preset['train']
val_transform = preset['val']

print("Small Dataset Configuration:")
print("=" * 50)
print(f"Training transform includes:")
for i, transform in enumerate(heavy_train_transform.transforms, 1):
    print(f"  {i}. {transform.__class__.__name__}")

print(f"\nTotal augmentations: {len(heavy_train_transform.transforms)}")
print("\nThis preset includes:")
print("- Rotation (up to 45 degrees)")
print("- Random brightness and contrast")
print("- Gaussian noise")
print("- Blur and sharpening")
print("- Elastic transformations")
print("- Grid distortion")
print("- Perspective changes")
print("- Coarse dropout (cutout)")
print("\nPerfect for datasets with:")
print("- Less than 1000 images per class")
print("- Need to prevent overfitting")
print("- Want to increase data diversity")

# Visualize augmentation effects
# show_augmentation_samples('sample_image.jpg', heavy_train_transform, n_samples=9)

### Example 5: Document OCR Preprocessing

In [ ]:
# Document OCR Example (Aksara Jawa OCR)
preset = PreprocessingPreset.document_ocr(img_size=(224, 224))
ocr_train_transform = preset['train']
ocr_val_transform = preset['val']

print("Document OCR Configuration:")
print("=" * 50)
print("Optimized for:")
print("- Character recognition")
print("- Handwritten text")
print("- Historical documents")
print("- Aksara Jawa, Aksara Sunda, etc.")
print("\nKey features:")
print("- CLAHE for contrast enhancement")
print("- Minimal rotation (< 10 degrees)")
print("- Perspective transformation")
print("- Noise injection for robustness")
print("- Sharpening for clear edges")

# Example: Process character images for OCR training
def prepare_ocr_dataset(root_dir, transform):
    """Prepare dataset for OCR training"""
    dataset = MultiClassDataset(root_dir, transform=transform)
    return dataset

# Usage
# train_ocr_dataset = prepare_ocr_dataset('dataset-aksara-jawa/train', ocr_train_transform)
# val_ocr_dataset = prepare_ocr_dataset('dataset-aksara-jawa/val', ocr_val_transform)

print("\n\nOCR Dataset Ready!")
print("Can be used for:")
print("- Training character classifier")
print("- Fine-tuning pretrained models")
print("- Creating OCR pipeline")

### Example 6: Batch Processing with Directory

In [ ]:
# Batch Processing Example

# Example 1: Process list of images
image_paths = ['img1.jpg', 'img2.jpg', 'img3.jpg', 'img4.jpg']
pipeline = get_validation_pipeline(img_size=(224, 224))

# Process and save to directory
# results = batch_process_images(image_paths, pipeline, output_dir='processed_images/')
print("Batch processing with list of paths")
print(f"Number of images: {len(image_paths)}")
print(f"Output directory: processed_images/")


# Example 2: Process entire directory
input_dir = 'dataset-aksara-jawa/prediction/prediction'
output_dir = 'dataset-aksara-jawa/prediction/processed'

# Custom preprocessing for prediction
predict_pipeline = A.Compose([
    A.Resize(224, 224),
    A.CLAHE(clip_limit=2.0),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Process directory
# results = process_directory(input_dir, predict_pipeline, output_dir, batch_size=32)
print("\n\nDirectory processing:")
print(f"Input: {input_dir}")
print(f"Output: {output_dir}")
print(f"Pipeline: CLAHE + Resize + Normalize")


# Example 3: Process with progress tracking
def process_with_tracking(image_paths, pipeline, output_dir):
    """Process images with progress bar"""
    results = []
    for i, img_path in enumerate(image_paths, 1):
        img = load_image(img_path)
        processed = pipeline(image=img)['image']
        
        # Save processed image
        output_path = os.path.join(output_dir, os.path.basename(img_path))
        cv2.imwrite(output_path, processed)
        
        results.append(processed)
        print(f"Processed {i}/{len(image_paths)}: {os.path.basename(img_path)}")
    
    return results

print("\n\nBatch processing supports:")
print("- Single image processing")
print("- List of images")
print("- Entire directory")
print("- Custom output directories")
print("- Progress tracking")

### Example 7: Visualization and Comparison

In [ ]:
# Visualization Examples

# Example 1: Compare preprocessing methods
sample_image = 'dataset-aksara-jawa/train/ba/ba_001.jpg'

preprocessing_methods = {
    'Original': lambda img: img,
    'CLAHE': lambda img: apply_clahe(img, clip_limit=2.0),
    'Histogram EQ': lambda img: histogram_equalization(img),
    'Gaussian Denoise': lambda img: denoise_image(img, method='gaussian'),
    'Bilateral Denoise': lambda img: denoise_image(img, method='bilateral'),
    'Sharpen': lambda img: sharpen_image(img)
}

# Show comparison
# show_preprocessing_comparison(sample_image, preprocessing_methods)
print("Preprocessing Comparison:")
print("=" * 50)
for method_name in preprocessing_methods.keys():
    print(f"- {method_name}")


# Example 2: Show augmentation samples
augmentation_pipeline = get_augmentation_pipeline(mode='heavy', img_size=(224, 224))

# Visualize multiple augmented versions
# show_augmentation_samples(sample_image, augmentation_pipeline, n_samples=9)
print("\n\nAugmentation Samples:")
print("=" * 50)
print("Shows 9 different augmented versions of the same image")
print("Useful for:")
print("- Understanding augmentation effects")
print("- Debugging augmentation pipeline")
print("- Visualizing data diversity")


# Example 3: Compare different augmentation intensities
print("\n\nAugmentation Intensity Comparison:")
print("=" * 50)

light_aug = get_augmentation_pipeline('light', (224, 224))
medium_aug = get_augmentation_pipeline('medium', (224, 224))
heavy_aug = get_augmentation_pipeline('heavy', (224, 224))

# Compare augmentation pipelines
aug_comparison = {
    'Light': light_aug,
    'Medium': medium_aug,
    'Heavy': heavy_aug
}

for name, pipeline in aug_comparison.items():
    print(f"{name}: {len(pipeline.transforms)} transforms")


# Example 4: Visualize before and after preprocessing
def visualize_pipeline_effect(image_path, pipeline):
    """Visualize effect of preprocessing pipeline"""
    # Load original
    original = load_image(image_path)
    
    # Apply pipeline
    processed = pipeline(image=original)['image']
    
    # Display side by side
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    axes[1].imshow(cv2.cvtColor(processed, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Processed')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

# Usage
# visualize_pipeline_effect(sample_image, augmentation_pipeline)
print("\n\nVisualization functions ready!")
print("Use these to inspect and debug your preprocessing pipeline")

### Example 8: PyTorch Integration

In [ ]:
# PyTorch Integration Examples

# Method 1: Using Albumentations (Recommended for training)
class AlbumentationsDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load image
        img = load_image(self.image_paths[idx])
        
        # Apply Albumentations transform
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented['image']
        
        # Convert to PyTorch tensor
        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        label = self.labels[idx]
        
        return img, label

# Setup with Albumentations preset
preset = PreprocessingPreset.multiclass_classification(img_size=(224, 224), n_classes=20)
# train_dataset = AlbumentationsDataset(train_paths, train_labels, transform=preset['train'])
# val_dataset = AlbumentationsDataset(val_paths, val_labels, transform=preset['val'])


# Method 2: Using torchvision transforms
from torchvision.datasets import ImageFolder

train_transforms = get_torch_train_transforms(img_size=(224, 224))
val_transforms = get_torch_val_transforms(img_size=(224, 224))

# Use with ImageFolder
# train_dataset = ImageFolder('dataset-aksara-jawa/train', transform=train_transforms)
# val_dataset = ImageFolder('dataset-aksara-jawa/val', transform=val_transforms)


# Method 3: Mixed approach (Albumentations + PyTorch)
class MixedDataset(Dataset):
    def __init__(self, image_paths, labels, albu_transform=None, torch_transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.albu_transform = albu_transform
        self.torch_transform = torch_transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        from PIL import Image
        
        # Load image
        img = load_image(self.image_paths[idx])
        
        # Apply Albumentations (strong augmentations)
        if self.albu_transform:
            augmented = self.albu_transform(image=img)
            img = augmented['image']
        
        # Convert to PIL for torchvision transforms
        img = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        
        # Apply torchvision transforms (normalization, to tensor)
        if self.torch_transform:
            img = self.torch_transform(img)
        
        label = self.labels[idx]
        return img, label


# Create DataLoaders
print("PyTorch Integration Methods:")
print("=" * 50)
print("\n1. Albumentations only (RECOMMENDED)")
print("   - Best for complex augmentations")
print("   - More flexible")
print("   - Supports geometric and color transforms")

print("\n2. Torchvision only")
print("   - Simple and straightforward")
print("   - Good for basic augmentations")
print("   - Native PyTorch integration")

print("\n3. Mixed approach")
print("   - Combines benefits of both")
print("   - Use Albumentations for augmentation")
print("   - Use torchvision for normalization")

# Example DataLoader setup
batch_size = 32
num_workers = 4

# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
# val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print("\n\nDataLoader Configuration:")
print(f"Batch size: {batch_size}")
print(f"Num workers: {num_workers}")
print("Ready for training!")

### Example 9: Medical Imaging Preprocessing

In [ ]:
# Medical Imaging Example (X-Ray, CT Scan, MRI)

preset = PreprocessingPreset.medical_imaging(img_size=(512, 512))
medical_train = preset['train']
medical_val = preset['val']

print("Medical Imaging Configuration:")
print("=" * 50)
print("Image size: 512x512 (higher resolution)")
print("\nOptimizations:")
print("- CLAHE for contrast enhancement")
print("- Minimal rotation (critical for medical diagnosis)")
print("- Brightness/contrast adjustments")
print("- Gaussian noise for robustness")
print("- NO aggressive geometric transforms")
print("\nIdeal for:")
print("- X-Ray classification")
print("- CT scan analysis")
print("- MRI processing")
print("- Pathology images")
print("- Retinal images")

# Custom medical preprocessing pipeline
medical_processor = ImagePreprocessor(img_size=(512, 512))
medical_processor.add_step(resize_image, size=(512, 512))
medical_processor.add_step(apply_clahe, clip_limit=2.5, tile_grid_size=(8, 8))
medical_processor.add_step(histogram_equalization)
medical_processor.add_step(denoise_image, method='non_local_means', h=10)

print("\n\nCustom Medical Pipeline:")
print("1. Resize to 512x512")
print("2. CLAHE (clip_limit=2.5)")
print("3. Histogram Equalization")
print("4. Non-local means denoising")

# Example: Chest X-Ray classification
class MedicalDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load grayscale medical image
        img = cv2.imread(self.image_paths[idx], cv2.IMREAD_GRAYSCALE)
        
        # Convert to 3 channels (required by most models)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        
        # Apply preprocessing
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented['image']
        
        # To tensor
        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        label = self.labels[idx]
        
        return img, label

print("\n\nMedical Dataset Ready!")
print("Supports:")
print("- Grayscale medical images")
print("- Automatic conversion to RGB")
print("- Specialized preprocessing")

### Example 10: Multi-Label Classification

In [ ]:
# Multi-Label Classification Example
# (Image can have multiple labels simultaneously)

preset = PreprocessingPreset.multilabel_classification(img_size=(224, 224))
multilabel_train = preset['train']
multilabel_val = preset['val']

print("Multi-Label Classification Configuration:")
print("=" * 50)
print("Use cases:")
print("- Image tagging (e.g., 'beach', 'sunset', 'people')")
print("- Medical diagnosis (multiple conditions)")
print("- Document classification (multiple categories)")
print("- Scene understanding (multiple objects)")

# Multi-label dataset example
class MultiLabelDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        """
        labels: list of lists, e.g., [[1, 0, 1, 0], [0, 1, 1, 0], ...]
        Each inner list represents binary labels for all classes
        """
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img = load_image(self.image_paths[idx])
        
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented['image']
        
        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        
        # Convert labels to tensor
        label = torch.FloatTensor(self.labels[idx])
        
        return img, label

# Example: Image with multiple tags
example_labels = [
    [1, 0, 1, 0, 1],  # image 1: has tags 0, 2, 4
    [0, 1, 1, 0, 0],  # image 2: has tags 1, 2
    [1, 1, 0, 1, 1],  # image 3: has tags 0, 1, 3, 4
]

print("\n\nExample Multi-Label Data:")
print("Image 1: Tags [0, 2, 4]")
print("Image 2: Tags [1, 2]")
print("Image 3: Tags [0, 1, 3, 4]")

print("\n\nKey differences from multi-class:")
print("- Uses BCE Loss instead of CrossEntropy")
print("- Sigmoid activation instead of Softmax")
print("- Can predict multiple labels per image")
print("- Labels are binary vectors, not single integers")

# Example model setup
print("\n\nModel Configuration:")
print("Loss function: BCEWithLogitsLoss")
print("Final activation: Sigmoid")
print("Metric: Hamming loss, F1-score, etc.")

## 10. Quick Reference & Best Practices

In [ ]:
quick_reference = """
╔════════════════════════════════════════════════════════════════╗
║            PREPROCESSING PIPELINE QUICK REFERENCE              ║
╚════════════════════════════════════════════════════════════════╝

📋 CHOOSING THE RIGHT PRESET
════════════════════════════════════════════════════════════════

┌─────────────────────┬──────────────────────────────────────────┐
│ Task                │ Use This Preset                          │
├─────────────────────┼──────────────────────────────────────────┤
│ Binary              │ PreprocessingPreset.binary_classification│
│ Multi-Class         │ PreprocessingPreset.multiclass_          │
│ Multi-Label         │ PreprocessingPreset.multilabel_          │
│ Small Dataset       │ PreprocessingPreset.small_dataset        │
│ Medical Images      │ PreprocessingPreset.medical_imaging      │
│ OCR/Documents       │ PreprocessingPreset.document_ocr         │
└─────────────────────┴──────────────────────────────────────────┘


🎯 AUGMENTATION INTENSITY
════════════════════════════════════════════════════════════════

Light (>5000 images/class):
  - Basic rotation (±15°)
  - Flip
  - Slight brightness/contrast

Medium (1000-5000 images/class):
  - Rotation (±30°)
  - Brightness/contrast
  - Blur/sharpen
  - Noise

Heavy (<1000 images/class):
  - Rotation (±45°)
  - Elastic deformation
  - Grid distortion
  - Cutout
  - Perspective


📐 IMAGE SIZE GUIDELINES
════════════════════════════════════════════════════════════════

224x224  → Standard, most pretrained models
299x299  → Inception models
384x384  → ViT models (some variants)
512x512  → Medical imaging, high detail
640x640  → Object detection


🔧 PREPROCESSING FUNCTIONS
════════════════════════════════════════════════════════════════

Basic:
  load_image(path)
  resize_image(img, size)
  normalize_image(img, mean, std)
  
Enhancement:
  apply_clahe(img, clip_limit=2.0)
  histogram_equalization(img)
  sharpen_image(img)
  
Denoising:
  denoise_image(img, method='gaussian')
  denoise_image(img, method='bilateral')
  denoise_image(img, method='non_local_means')


💡 BEST PRACTICES
════════════════════════════════════════════════════════════════

1. Always use different augmentation for train/val
   ✓ train: heavy augmentation
   ✓ val: only resize + normalize

2. Match normalization to pretrained model
   ✓ ImageNet: mean=[0.485, 0.456, 0.406]
                std=[0.229, 0.224, 0.225]

3. Start with preset, customize if needed
   preset = PreprocessingPreset.multiclass_classification()
   # Modify if necessary

4. Use Albumentations for training
   ✓ More augmentation options
   ✓ Better performance
   ✓ Easy to customize

5. Visualize before training
   show_augmentation_samples('image.jpg', pipeline)


⚡ PERFORMANCE TIPS
════════════════════════════════════════════════════════════════

1. Use num_workers in DataLoader
   DataLoader(dataset, num_workers=4)

2. Cache preprocessed images if possible
   processor.process_batch(images, save_dir='cache/')

3. Use pin_memory for GPU training
   DataLoader(dataset, pin_memory=True)

4. Batch process for efficiency
   batch_process_images(paths, pipeline, batch_size=32)


🐛 COMMON ISSUES & SOLUTIONS
════════════════════════════════════════════════════════════════

Issue: Model not learning
→ Check augmentation is not too heavy
→ Verify labels are correct
→ Ensure normalization matches pretrained model

Issue: Overfitting
→ Increase augmentation intensity
→ Use smaller model
→ Add more dropout

Issue: Poor validation accuracy
→ Reduce augmentation on validation
→ Check for data leakage
→ Verify preprocessing consistency


📊 MONITORING
════════════════════════════════════════════════════════════════

Always monitor:
- Training vs validation loss
- Training vs validation accuracy
- Check overfitting signs
- Visualize predictions
- Inspect augmented samples


🚀 QUICK START TEMPLATE
════════════════════════════════════════════════════════════════

# 1. Choose preset
preset = PreprocessingPreset.multiclass_classification(
    img_size=(224, 224), 
    n_classes=20
)

# 2. Create dataset
dataset = YourDataset(
    image_paths=paths,
    labels=labels,
    transform=preset['train']
)

# 3. Create dataloader
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4
)

# 4. Train!
for images, labels in loader:
    # Your training code here
    pass
"""

print(quick_reference)

### Summary & Reusability

In [ ]:
summary = """
╔════════════════════════════════════════════════════════════════╗
║         PREPROCESSING TEMPLATE - COMPLETE & REUSABLE           ║
╚════════════════════════════════════════════════════════════════╝

✅ WHAT'S INCLUDED
════════════════════════════════════════════════════════════════

📦 Section 1: Setup & Dependencies
   - All required imports
   - Environment configuration
   
📦 Section 2: Basic Preprocessing
   - load_image, resize_image, normalize_image
   - denormalize_image (for visualization)
   
📦 Section 3: Advanced Preprocessing
   - CLAHE (contrast enhancement)
   - Histogram equalization
   - Multiple denoising methods
   - Image sharpening
   
📦 Section 4: Augmentation Pipelines
   - Light, Medium, Heavy presets
   - Validation pipeline
   - Customizable parameters
   
📦 Section 5: PyTorch Transforms
   - Training transforms
   - Validation transforms
   - Full torchvision integration
   
📦 Section 6: Preprocessing Presets
   - Binary classification
   - Multi-class classification
   - Multi-label classification
   - Small dataset optimization
   - Medical imaging
   - Document OCR
   
📦 Section 7: Custom Pipeline Class
   - ImagePreprocessor
   - Add custom steps
   - Process single/batch
   
📦 Section 8: Batch Processing
   - Process multiple images
   - Process directories
   - Save results automatically
   
📦 Section 9: Visualization
   - Compare preprocessing methods
   - Show augmentation samples
   - Debug pipeline effects
   
📦 Section 10: Usage Examples
   - 10 complete examples
   - All common scenarios
   - Ready-to-use code


🎯 HOW TO USE THIS TEMPLATE
════════════════════════════════════════════════════════════════

For NEW projects:
1. Copy this entire notebook
2. Run Section 1 (setup)
3. Choose appropriate preset (Section 6)
4. Create your dataset class (Section 9 examples)
5. Start training!

For EXISTING projects:
1. Copy only the functions you need
2. Integrate into your codebase
3. Customize as needed

For QUICK experiments:
1. Use PreprocessingPreset directly
2. No need to understand internal details
3. Just select the right preset for your task


📚 REUSABILITY FEATURES
════════════════════════════════════════════════════════════════

✓ Modular Functions
  - Each function is independent
  - Copy-paste friendly
  - Well documented

✓ Flexible Presets
  - Cover 90% of common cases
  - Easy to modify
  - Consistent API

✓ Clear Examples
  - Real-world scenarios
  - Complete code
  - Explained logic

✓ Multiple Integration Methods
  - Albumentations
  - torchvision
  - Custom pipelines
  - All compatible


🔄 MAINTENANCE & UPDATES
════════════════════════════════════════════════════════════════

This template is designed to be:
- Version-agnostic (works with latest libraries)
- Easy to update (modular structure)
- Expandable (add new functions easily)
- Backward compatible (old code still works)


💾 SAVE THIS TEMPLATE
════════════════════════════════════════════════════════════════

Keep this notebook as reference for:
- Future projects
- Team members
- Quick prototyping
- Documentation
- Teaching/learning


🎓 LEARNING PATH
════════════════════════════════════════════════════════════════

Beginner:
→ Start with PreprocessingPreset
→ Use provided examples
→ Don't modify anything

Intermediate:
→ Understand each function
→ Customize presets
→ Mix and match techniques

Advanced:
→ Create custom pipelines
→ Add new preprocessing methods
→ Optimize for specific cases


🚀 NEXT STEPS
════════════════════════════════════════════════════════════════

1. Test on your data
   - Load a few sample images
   - Visualize preprocessing effects
   - Verify augmentations look good

2. Integrate with training
   - Create dataset class
   - Setup data loaders
   - Train baseline model

3. Iterate and improve
   - Monitor training metrics
   - Adjust augmentation intensity
   - Fine-tune preprocessing


📝 NOTES
════════════════════════════════════════════════════════════════

- Always visualize before training
- Start simple, add complexity gradually
- Document your changes
- Keep original template intact
- Share improvements with team


═══════════════════════════════════════════════════════════════

                    HAPPY PREPROCESSING! 🎨
                    
═══════════════════════════════════════════════════════════════
"""

print(summary)

# Print final statistics
print("\n\n" + "="*70)
print("TEMPLATE STATISTICS")
print("="*70)
print(f"✓ Basic Functions: 7")
print(f"✓ Advanced Functions: 4")
print(f"✓ Augmentation Pipelines: 4")
print(f"✓ Preprocessing Presets: 6")
print(f"✓ Dataset Classes: 5")
print(f"✓ Complete Examples: 10")
print(f"✓ Visualization Tools: 3")
print("="*70)
print("READY TO USE! 🚀")
print("="*70)